# GDS capacitance with zero-thickness electrodes

This notebook creates a separate COMSOL 6.1 electrostatics model. The patterned GDS electrodes are imported as two-dimensional shell faces at `z = 0`, and the full-chip buried ground plane is the zero-thickness interface at `z = -3 um` between two SiO2 blocks. No conductor volume is created.

From top to bottom, the modeled stack is: zero-thickness patterned electrodes, 3 um SiO2, zero-thickness buried GND, 12 um SiO2, and 675 um silicon. This perfect-conductor boundary approximation removes all 0.5 um metal sidewalls. It intentionally neglects finite-thickness sidewall fringing.

The notebook configures geometry, validated named selections, materials, Electrostatics, a sequential swept-oxide mesh, free-tetrahedral silicon and air meshes, a stationary study, and two independent capacitance evaluations. It does not build the mesh or solve.

In [ ]:
from pathlib import Path
import numpy as np
import mph
from jpype import JArray
from jpype.types import JBoolean, JInt

# This Python environment must have MPh installed and COMSOL 6.1 available.
client = mph.start(version='6.1')
print('COMSOL version:', client.version)
model = client.create('gds_buried_ground_zero_thickness')
j = model.java


## 1. Configuration

All geometry and mesh dimensions are expressed in micrometers. The RF electrode is identified by the physical xy bounding box measured for RF domain 14 in the original finite-thickness model, rather than by unstable COMSOL entity numbers.

In [ ]:
GDS_FILE = Path(r'C:\Users\Administrator\Downloads\COMSOL_MODEL_V3.gds')
GDS_CELL = 'UNNAMED_12'
ELECTRODE_LAYER = 'LAYER8'

TOP_ELECTRODE_Z_UM = 0.0
TOP_OXIDE_THICKNESS_UM = 3.0
BURIED_GND_Z_UM = TOP_ELECTRODE_Z_UM - TOP_OXIDE_THICKNESS_UM
BOTTOM_OXIDE_THICKNESS_UM = 12.0
BOTTOM_OXIDE_Z_MIN_UM = BURIED_GND_Z_UM - BOTTOM_OXIDE_THICKNESS_UM
SILICON_THICKNESS_UM = 675.0
SILICON_Z_MAX_UM = BOTTOM_OXIDE_Z_MIN_UM
SILICON_Z_MIN_UM = SILICON_Z_MAX_UM - SILICON_THICKNESS_UM

CHIP_LENGTH_UM = 11976.0
CHIP_WIDTH_UM = 5846.51
AIR_Z_MIN_UM = -1000.0
AIR_Z_MAX_UM = 500.0
AIR_XY_PADDING_UM = 500.0

# xmin, xmax, ymin, ymax for the original RF electrode.
RF_REFERENCE_XY_BBOX_UM = np.array([
    9951.12988, 15276.0, -170.371994, 946.627991
])
RF_BBOX_TOLERANCE_UM = 1e-3
EXPECTED_PATTERNED_FACE_COUNT = 23

MESH_RF = {'hmax': 20.0, 'hmin': 1.0, 'growth': 1.40}
MESH_PATTERNED_GND = {'hmax': 40.0, 'hmin': 2.0, 'growth': 1.50}
MESH_BURIED_GND = {'hmax': 50.0, 'hmin': 10.0, 'growth': 1.50}
MESH_SIO2 = {'hmax': 25.0, 'hmin': 10.0, 'growth': 1.40}
MESH_SILICON = {'hmax': 150.0, 'hmin': 15.0, 'growth': 1.50}
MESH_AIR = {'hmax': 500.0, 'hmin': 25.0, 'growth': 1.60}

if not GDS_FILE.is_file():
    raise FileNotFoundError(f'GDS file not found: {GDS_FILE}')
if GDS_FILE.suffix.lower() != '.gds':
    raise ValueError('The layout file must have a .gds extension.')
GDS_FILE = GDS_FILE.resolve()

print('GDS:', GDS_FILE)
print('Zero-thickness patterned electrode z [um]:', TOP_ELECTRODE_Z_UM)
print('3 um SiO2 [um]:', BURIED_GND_Z_UM, TOP_ELECTRODE_Z_UM)
print('Zero-thickness buried GND z [um]:', BURIED_GND_Z_UM)
print('12 um SiO2 [um]:', BOTTOM_OXIDE_Z_MIN_UM, BURIED_GND_Z_UM)
print('Silicon [um]:', SILICON_Z_MIN_UM, SILICON_Z_MAX_UM)
assert np.isclose(TOP_ELECTRODE_Z_UM-BURIED_GND_Z_UM, 3.0)
assert np.isclose(BURIED_GND_Z_UM-BOTTOM_OXIDE_Z_MIN_UM, 12.0)


## 2. Import `LAYER8` as zero-thickness faces

The decisive setting is `importtype = 'shell'`. In a 3D ECAD import this produces faces instead of solid metal domains. No layer height is assigned. Resulting selections are enabled so the imported faces can be recovered after Form Union.

In [ ]:
j.component().create('comp1', JBoolean(True))
comp = j.component('comp1')
comp.geom().create('geom1', 3)
geom = comp.geom('geom1')

geom.create('imp1', 'Import')
imp = geom.feature('imp1')
imp.label('Zero-thickness patterned electrodes')
imp.set('filename', str(GDS_FILE))
imp.set('updategeomunit', JBoolean(True))
imp.set('grouping', 'layer')
imp.set('importtype', 'shell')
imp.set('manualelevation', 'on')
imp.set('intbnd', 'on')
imp.set('findarcs', 'auto')
imp.set('repairgeom', 'on')
imp.set('repairtoltype', 'auto')
imp.set('selresult', 'on')
imp.set('selresultshow', 'all')
imp.set('sellayer', 'on')
imp.set('sellayershow', 'all')

ecad_type = str(imp.getString('ecadtype')).lower()
if ecad_type != 'gds':
    raise RuntimeError(f'COMSOL identified ECAD type {ecad_type!r}, not GDS.')
layer_table = np.asarray(imp.getStringMatrix('layerprop'), dtype=object)
if layer_table.size == 0:
    raise RuntimeError('COMSOL found no GDS layers.')
if GDS_CELL:
    imp.set('cell', GDS_CELL)

layer_names = [str(row[0]).upper() for row in layer_table]
import_flags = [
    'on' if name == ELECTRODE_LAYER else 'off'
    for name in layer_names
]
if not any(flag == 'on' for flag in import_flags):
    raise RuntimeError(f'{ELECTRODE_LAYER} is absent; layers={layer_names}')
imp.set('importlayer', import_flags)
imp.set('elevation', [
    f'{TOP_ELECTRODE_Z_UM}[um]' if flag == 'on' else '0[um]'
    for flag in import_flags
])

try:
    imp.importData()
    geom.run('imp1')
except Exception as exc:
    raise RuntimeError(
        'GDS shell import failed. Check the ECAD Import Module license and file.'
    ) from exc

object_names = list(imp.objectNames())
if not object_names:
    raise RuntimeError('The GDS shell import produced no objects.')
measure_import = geom.measure()
measure_import.selection().set(object_names)
electrode_bbox = np.asarray(measure_import.getBoundingBox(), dtype=float)
exmin, exmax, eymin, eymax, ezmin, ezmax = electrode_bbox
if not np.allclose([ezmin, ezmax], [TOP_ELECTRODE_Z_UM]*2, atol=1e-9):
    raise RuntimeError(f'Imported shells have unexpected z bounds: {electrode_bbox}')
print('Geometry unit:', geom.lengthUnit())
print('Patterned-shell bounding box [um]:', electrode_bbox)


## 3. Construct the dielectric stack and air

There is no buried-metal block. The retained interface between the two oxide blocks is the buried electrode. The imported shell is also not subtracted from the air because it has no volume. Form Union must retain interior boundaries and imprint the shell outlines into the oxide-air interface.

In [ ]:
chip_xmax = exmax
chip_xmin = chip_xmax - CHIP_LENGTH_UM
chip_ycenter = 0.5 * (eymin + eymax)
chip_ymin = chip_ycenter - CHIP_WIDTH_UM/2
chip_ymax = chip_ymin + CHIP_WIDTH_UM

def add_block(tag, label, position_um, size_um):
    geom.create(tag, 'Block')
    block = geom.feature(tag)
    block.label(label)
    block.set('base', 'corner')
    block.set('pos', [f'{value:.12g}' for value in position_um])
    block.set('size', [f'{value:.12g}' for value in size_um])
    block.set('selresult', 'on')
    block.set('selresultshow', 'all')
    return block

add_block(
    'oxide_top', '3 um top SiO2',
    [chip_xmin, chip_ymin, BURIED_GND_Z_UM],
    [CHIP_LENGTH_UM, CHIP_WIDTH_UM, TOP_OXIDE_THICKNESS_UM],
)
add_block(
    'oxide_bottom', '12 um bottom SiO2',
    [chip_xmin, chip_ymin, BOTTOM_OXIDE_Z_MIN_UM],
    [CHIP_LENGTH_UM, CHIP_WIDTH_UM, BOTTOM_OXIDE_THICKNESS_UM],
)
add_block(
    'silicon', '675 um silicon',
    [chip_xmin, chip_ymin, SILICON_Z_MIN_UM],
    [CHIP_LENGTH_UM, CHIP_WIDTH_UM, SILICON_THICKNESS_UM],
)

air_xmin = chip_xmin - AIR_XY_PADDING_UM
air_ymin = chip_ymin - AIR_XY_PADDING_UM
add_block(
    'air_box', 'Exterior air box',
    [air_xmin, air_ymin, AIR_Z_MIN_UM],
    [CHIP_LENGTH_UM+2*AIR_XY_PADDING_UM,
     CHIP_WIDTH_UM+2*AIR_XY_PADDING_UM,
     AIR_Z_MAX_UM-AIR_Z_MIN_UM],
)

geom.create('dif_air', 'Difference')
dif_air = geom.feature('dif_air')
dif_air.label('Air minus dielectric stack')
dif_air.selection('input').set(['air_box'])
dif_air.selection('input2').set([
    'oxide_top', 'oxide_bottom', 'silicon'
])
dif_air.set('keepsubtract', 'on')
dif_air.set('intbnd', 'on')
dif_air.set('selresult', 'on')
dif_air.set('selresultshow', 'all')

# Form Union makes the zero-thickness shell faces part of the final geometry.
# In COMSOL 6.1, Finalize has no intbnd property; the union retains the
# boundaries contributed by its participating geometry objects.
geom.feature('fin').set('action', 'union')
geom.run()

final_bbox = np.asarray(geom.getBoundingBox(), dtype=float)
if not np.allclose(final_bbox[[4, 5]], [AIR_Z_MIN_UM, AIR_Z_MAX_UM]):
    raise RuntimeError(f'Unexpected final z bounds: {final_bbox[[4,5]]}')
print('Chip x [um]:', chip_xmin, chip_xmax)
print('Chip y [um]:', chip_ymin, chip_ymax)
print('Full model bounding box [um]:', final_bbox)


## 4. Recover and validate dielectric and electrode entities

The shell import must produce boundary entities adjacent to the dielectric solution domains. The code stops if the shells remain orphaned, if RF cannot be matched uniquely, or if the complete buried oxide-oxide interface cannot be identified. Entity IDs are then stored in explicit named selections.

In [ ]:
def selection_ids(tag, dimension):
    return sorted(
        int(v) for v in comp.selection(tag).entities(JInt(dimension))
    )

selection_tags = [str(tag) for tag in comp.selection().tags()]
layer_shell_matches = [
    tag for tag in selection_tags
    if 'imp1' in tag and ELECTRODE_LAYER.lower() in tag.lower()
    and tag.lower().endswith('_bnd')
]
if len(layer_shell_matches) == 1:
    shell_selection_tag = layer_shell_matches[0]
else:
    overall_tag = 'geom1_imp1_bnd'
    if overall_tag not in selection_tags:
        raise RuntimeError(
            'Cannot uniquely find the imported shell-boundary selection. '
            f'Layer matches: {layer_shell_matches}'
        )
    shell_selection_tag = overall_tag

top_electrode_face_ids = selection_ids(shell_selection_tag, 2)
if not top_electrode_face_ids:
    raise RuntimeError('The imported shell selection is empty.')
if len(top_electrode_face_ids) != EXPECTED_PATTERNED_FACE_COUNT:
    print(
        'Note: Form Union split the original 23 conductor polygons into '
        f'{len(top_electrode_face_ids)} shell faces at material/perimeter edges.'
    )

air_ids = selection_ids('geom1_dif_air_dom', 3)
top_oxide_ids = selection_ids('geom1_oxide_top_dom', 3)
bottom_oxide_ids = selection_ids('geom1_oxide_bottom_dom', 3)
silicon_ids = selection_ids('geom1_silicon_dom', 3)
sio2_ids = sorted(set(top_oxide_ids + bottom_oxide_ids))
dielectric_ids = sorted(set(air_ids + sio2_ids + silicon_ids))

def adjacent_boundaries(domain_ids):
    result = set()
    for domain_id in domain_ids:
        result.update(int(v) for v in geom.getAdj(
            JInt(3), JInt(2), JInt(domain_id)
        ))
    return sorted(result)

boundary_measure = comp.measure()
boundary_measure.selection().geom('geom1', JInt(2))
def boundary_bbox(boundary_id):
    boundary_measure.selection().set(JArray(JInt)([int(boundary_id)]))
    return np.asarray(boundary_measure.getBoundingBox(), dtype=float)

# Confirm that every shell face was imprinted into a solved dielectric boundary.
dielectric_boundary_ids = set(adjacent_boundaries(dielectric_ids))
orphan_shell_faces = sorted(
    set(top_electrode_face_ids) - dielectric_boundary_ids
)
if orphan_shell_faces:
    raise RuntimeError(
        'Imported shell faces were not imprinted into the dielectric geometry: '
        f'{orphan_shell_faces}. Check Form Union and coincident z positions.'
    )

# Every patterned shell must be planar and located at z = 0.
bad_z_faces = []
face_bboxes = {}
for boundary_id in top_electrode_face_ids:
    bbox = boundary_bbox(boundary_id)
    face_bboxes[boundary_id] = bbox
    if not np.allclose(
        bbox[[4, 5]], [TOP_ELECTRODE_Z_UM]*2, atol=1e-6, rtol=0.0
    ):
        bad_z_faces.append(boundary_id)
if bad_z_faces:
    raise RuntimeError(f'Electrode faces are not at z=0: {bad_z_faces}')

rf_boundary_ids = [
    boundary_id for boundary_id, bbox in face_bboxes.items()
    if np.allclose(
        bbox[:4], RF_REFERENCE_XY_BBOX_UM,
        rtol=0.0, atol=RF_BBOX_TOLERANCE_UM
    )
]
if len(rf_boundary_ids) != 1:
    errors = {
        boundary_id: float(np.max(np.abs(
            bbox[:4]-RF_REFERENCE_XY_BBOX_UM
        )))
        for boundary_id, bbox in face_bboxes.items()
    }
    raise RuntimeError(
        f'RF shell match is not unique: {rf_boundary_ids}. '
        f'Maximum bbox errors [um]: {errors}'
    )
patterned_gnd_boundary_ids = sorted(
    set(top_electrode_face_ids)-set(rf_boundary_ids)
)

# The buried plane is the complete shared horizontal oxide interface.
shared_oxide_boundaries = (
    set(adjacent_boundaries(top_oxide_ids))
    & set(adjacent_boundaries(bottom_oxide_ids))
)
buried_gnd_boundary_ids = []
for boundary_id in shared_oxide_boundaries:
    bbox = boundary_bbox(boundary_id)
    if np.allclose(
        bbox[[4, 5]], [BURIED_GND_Z_UM]*2, atol=1e-6, rtol=0.0
    ):
        buried_gnd_boundary_ids.append(boundary_id)
buried_gnd_boundary_ids = sorted(buried_gnd_boundary_ids)
if not buried_gnd_boundary_ids:
    raise RuntimeError('No shared oxide boundary was found at the buried-GND z.')

gnd_boundary_ids = sorted(
    set(patterned_gnd_boundary_ids) | set(buried_gnd_boundary_ids)
)
if set(rf_boundary_ids) & set(gnd_boundary_ids):
    raise RuntimeError('RF and GND boundary selections overlap.')

def explicit_selection(tag, label, dimension, entity_ids):
    selection = comp.selection().create(tag, 'Explicit')
    selection.label(label)
    selection.geom('geom1', JInt(dimension))
    selection.set(JArray(JInt)(list(entity_ids)))
    return selection

explicit_selection('AIR', 'AIR', 3, air_ids)
explicit_selection('SIO2_TOP', '3 um top SiO2', 3, top_oxide_ids)
explicit_selection('SIO2_BOTTOM', '12 um bottom SiO2', 3, bottom_oxide_ids)
explicit_selection('SIO2', 'All SiO2', 3, sio2_ids)
explicit_selection('SILICON', 'Silicon', 3, silicon_ids)
explicit_selection(
    'DIELECTRICS', 'AIR + all SiO2 + Silicon', 3, dielectric_ids
)
explicit_selection('RF', 'Zero-thickness RF face', 2, rf_boundary_ids)
explicit_selection(
    'GND_PATTERNED', 'Zero-thickness patterned GND faces', 2,
    patterned_gnd_boundary_ids
)
explicit_selection(
    'GND_BURIED', 'Zero-thickness buried GND plane', 2,
    buried_gnd_boundary_ids
)
explicit_selection('GND', 'All zero-thickness GND faces', 2, gnd_boundary_ids)

def horizontal_faces(domain_ids, z_um, tolerance_um=1e-6):
    matches = []
    for boundary_id in adjacent_boundaries(domain_ids):
        bbox = boundary_bbox(boundary_id)
        if (abs(bbox[4]-z_um) <= tolerance_um and
                abs(bbox[5]-z_um) <= tolerance_um):
            matches.append(boundary_id)
    if not matches:
        raise RuntimeError(
            f'No horizontal faces found at z={z_um} um for {domain_ids}.'
        )
    return sorted(matches)

top_oxide_top_faces = horizontal_faces(top_oxide_ids, TOP_ELECTRODE_Z_UM)
top_oxide_bottom_faces = horizontal_faces(top_oxide_ids, BURIED_GND_Z_UM)
bottom_oxide_top_faces = horizontal_faces(bottom_oxide_ids, BURIED_GND_Z_UM)
bottom_oxide_bottom_faces = horizontal_faces(
    bottom_oxide_ids, BOTTOM_OXIDE_Z_MIN_UM
)
# The specified chip is approximately 0.98 um narrower in y than the GDS
# bounding box. Form Union therefore splits narrow conductor overhangs into
# separate shell faces that touch air but not the oxide. Accept only such
# true perimeter overhangs; a shell projecting over the chip must be imprinted
# into the top-oxide boundary.
top_source_set = set(top_oxide_top_faces)
electrode_faces_on_oxide = sorted(
    set(top_electrode_face_ids) & top_source_set
)
electrode_overhang_faces = sorted(
    set(top_electrode_face_ids) - top_source_set
)
if not electrode_faces_on_oxide:
    raise RuntimeError('No GDS shell face was imprinted into the top oxide.')
if not set(rf_boundary_ids).issubset(top_source_set):
    raise RuntimeError('The RF shell is not part of the top oxide surface.')

def projected_chip_overlap_widths(bbox):
    overlap_x = max(
        0.0, min(bbox[1], chip_xmax)-max(bbox[0], chip_xmin)
    )
    overlap_y = max(
        0.0, min(bbox[3], chip_ymax)-max(bbox[2], chip_ymin)
    )
    return overlap_x, overlap_y

bad_unimprinted_faces = [
    boundary_id for boundary_id in electrode_overhang_faces
    if min(projected_chip_overlap_widths(face_bboxes[boundary_id])) > 1e-6
]
if bad_unimprinted_faces:
    raise RuntimeError(
        'These shell faces project over the chip but were not imprinted into '
        f'the oxide surface: {bad_unimprinted_faces}'
    )
if set(top_oxide_bottom_faces) != set(buried_gnd_boundary_ids):
    raise RuntimeError('Top-oxide target does not equal the buried-GND interface.')
if set(bottom_oxide_top_faces) != set(buried_gnd_boundary_ids):
    raise RuntimeError('Bottom-oxide source does not equal the buried-GND interface.')

explicit_selection(
    'TOP_OXIDE_SOURCE', 'Top oxide source faces', 2, top_oxide_top_faces
)
explicit_selection(
    'TOP_OXIDE_TARGET', 'Top oxide target / buried GND', 2,
    top_oxide_bottom_faces
)
explicit_selection(
    'BOTTOM_OXIDE_SOURCE', 'Bottom oxide source / buried GND', 2,
    bottom_oxide_top_faces
)
explicit_selection(
    'BOTTOM_OXIDE_TARGET', 'Bottom oxide target faces', 2,
    bottom_oxide_bottom_faces
)

print('Imported shell selection:', shell_selection_tag)
print('Patterned electrode faces:', len(top_electrode_face_ids))
print('RF face:', rf_boundary_ids, face_bboxes[rf_boundary_ids[0]])
print('Patterned GND face count:', len(patterned_gnd_boundary_ids))
print('Electrode faces on oxide:', len(electrode_faces_on_oxide))
print('Valid perimeter-overhang faces:', electrode_overhang_faces)
print('Buried GND interface:', buried_gnd_boundary_ids)
print('AIR/SIO2/Si domains:', air_ids, sio2_ids, silicon_ids)


## 5. Materials and Electrostatics

Air, both oxides, and silicon are the only physics domains. Relative permittivities are 1.0, 3.9, and 11.7. A 1 V boundary Terminal is applied to RF; every other patterned face and the buried oxide interface are Ground. COMSOL's terminal-charge variable and stored energy provide independent capacitance estimates.

In [ ]:
def add_isotropic_material(tag, label, epsr, selection_tag):
    material = comp.material().create(tag, 'Common')
    material.label(label)
    material.selection().named(selection_tag)
    tensor = [
        str(epsr), '0', '0',
        '0', str(epsr), '0',
        '0', '0', str(epsr),
    ]
    material.propertyGroup('def').set('relpermittivity', tensor)
    return material

add_isotropic_material('mat_air', 'Air', 1.0, 'AIR')
add_isotropic_material('mat_sio2', 'SiO2', 3.9, 'SIO2')
add_isotropic_material('mat_silicon', 'Silicon (dielectric)', 11.7, 'SILICON')

j.param().set('Vrf', '1[V]', 'RF terminal voltage')
comp.physics().create('es', 'Electrostatics', 'geom1')
es = comp.physics('es')
es.selection().named('DIELECTRICS')

es_tags = [str(tag) for tag in es.feature().tags()]
if 'ccn1' not in es_tags:
    raise RuntimeError(f'Default Charge Conservation is missing: {es_tags}')
es.feature('ccn1').label('Charge Conservation: all dielectric domains')
if 'zerochg1' in es_tags:
    es.feature('zerochg1').label('Zero charge on remaining exterior boundaries')

es.create('gnd1', 'Ground', JInt(2))
es.feature('gnd1').label('Patterned GND + zero-thickness buried plane')
es.feature('gnd1').selection().named('GND')

es.create('term1', 'Terminal', JInt(2))
terminal = es.feature('term1')
terminal.label('Zero-thickness RF terminal: 1 V')
terminal.selection().named('RF')
terminal.set('TerminalType', 'Voltage')
terminal.set('TerminalName', '1')
terminal.set('V0', 'Vrf')
print('Electrostatics configured; no solve has been run.')


## 6. Sequential oxide sweeps, tetrahedral bulk mesh, and study

The GDS-partitioned top surface is triangulated first. That mesh is swept through the 3 um oxide to the buried-GND interface, and the resulting interface mesh is immediately reused as the source of the 12 um oxide sweep. This avoids creating two incompatible meshes on the same zero-thickness ground plane. Silicon and air use free tetrahedra.

Requested surface sizes are RF 20/1 um and patterned GND 40/2 um. The oxide setting is 25/10 um, silicon is 150/15 um, and air is 500/25 um. The nominal buried-plane setting is 50/10 um, but its conforming mesh is inherited from the finer top-oxide sweep and therefore can be finer. Because the buried plane has no face adjacent to air, it is not added as an air-mesh size selection.

This cell creates the mesh sequence and stationary study but deliberately calls neither `mesh.run()` nor `study.run()`.

In [ ]:
comp.mesh().create('mesh1')
mesh = comp.mesh('mesh1')
mesh.label('Zero-thickness sequential swept/tetrahedral mesh')

def add_mesh_size(parent, tag, label, selection_tag, settings):
    parent.create(tag, 'Size')
    size = parent.feature(tag)
    size.label(label)
    size.selection().named(selection_tag)
    size.set('custom', JBoolean(True))
    size.set('hmax', f"{settings['hmax']}[um]")
    size.set('hmin', f"{settings['hmin']}[um]")
    size.set('hgrad', str(settings['growth']))
    return size

def add_swept_domain(tag, label, domain_selection, source_selection,
                     target_selection, settings, layers):
    mesh.create(tag, 'Sweep')
    sweep = mesh.feature(tag)
    sweep.label(label)
    sweep.selection().named(domain_selection)
    sweep.selection('sourceface').named(source_selection)
    sweep.selection('targetface').named(target_selection)
    sweep.set('facemethod', 'tri')
    sweep.set('sweeppath', 'straight')
    add_mesh_size(
        sweep, 'size1', f'{label}: lateral size', domain_selection, settings
    )
    sweep.create('dist1', 'Distribution')
    distribution = sweep.feature('dist1')
    distribution.label(f'{layers} elements through thickness')
    distribution.selection().named(domain_selection)
    distribution.set('numelem', JInt(layers))
    return sweep

# 1. Triangulate the complete top-oxide surface, including GDS partitions.
mesh.create('ftri_top', 'FreeTri')
ftri_top = mesh.feature('ftri_top')
ftri_top.label('Triangular source mesh on zero-thickness top electrodes')
ftri_top.selection().named('TOP_OXIDE_SOURCE')
add_mesh_size(
    ftri_top, 'size_sio2', 'Exposed top SiO2: 25/10 um',
    'TOP_OXIDE_SOURCE', MESH_SIO2
)
add_mesh_size(
    ftri_top, 'size_gnd', 'Patterned GND: 40/2 um',
    'GND_PATTERNED', MESH_PATTERNED_GND
)
add_mesh_size(
    ftri_top, 'size_rf', 'RF: 20/1 um', 'RF', MESH_RF
)

# 2. Sweep the top oxide. Its target is the buried ground interface.
add_swept_domain(
    'swe_top_oxide', 'Sweep 3 um top SiO2', 'SIO2_TOP',
    'TOP_OXIDE_SOURCE', 'TOP_OXIDE_TARGET', MESH_SIO2, 3
)

# 3. Reuse that target mesh as the source of the lower oxide sweep.
add_swept_domain(
    'swe_bottom_oxide', 'Sweep 12 um bottom SiO2', 'SIO2_BOTTOM',
    'BOTTOM_OXIDE_SOURCE', 'BOTTOM_OXIDE_TARGET', MESH_SIO2, 4
)

# 4. Fill the thick silicon substrate with free tetrahedra.
mesh.create('ftet_silicon', 'FreeTet')
ftet_silicon = mesh.feature('ftet_silicon')
ftet_silicon.label('Free tetrahedra in silicon')
ftet_silicon.selection().named('SILICON')
add_mesh_size(
    ftet_silicon, 'size_si', 'Silicon: 150/15 um',
    'SILICON', MESH_SILICON
)

# 5. Fill air last so it conforms to all existing chip-surface meshes.
mesh.create('ftet_air', 'FreeTet')
ftet_air = mesh.feature('ftet_air')
ftet_air.label('Free tetrahedra in exterior air only')
ftet_air.selection().named('AIR')
add_mesh_size(ftet_air, 'size_air', 'Air: 500/25 um', 'AIR', MESH_AIR)
add_mesh_size(
    ftet_air, 'size_patterned_gnd', 'Patterned GND: 40/2 um',
    'GND_PATTERNED', MESH_PATTERNED_GND
)
add_mesh_size(
    ftet_air, 'size_rf', 'RF: 20/1 um', 'RF', MESH_RF
)

j.study().create('std1')
j.study('std1').label('Stationary zero-thickness capacitance study')
j.study('std1').create('stat', 'Stationary')

comp.cpl().create('intWop', 'Integration', 'geom1')
comp.cpl('intWop').label('Electrostatic energy over all dielectrics')
comp.cpl('intWop').set('opname', 'intWop')
comp.cpl('intWop').selection().named('DIELECTRICS')

j.result().numerical().create('gevCap', 'EvalGlobal')
gev = j.result().numerical('gevCap')
gev.label('Zero-thickness capacitance checks')
gev.set('expr', [
    '2*intWop(es.We)/Vrf^2',
    'abs(es.Q0_1)/abs(Vrf)',
    'abs(2*intWop(es.We)/Vrf^2-abs(es.Q0_1)/abs(Vrf))/'
    'max(abs(es.Q0_1)/abs(Vrf),1e-30[F])',
])
gev.set('unit', ['F', 'F', '1'])
gev.set('descr', [
    'Capacitance from stored energy',
    'Capacitance from COMSOL terminal charge',
    'Relative disagreement: energy versus terminal charge',
])

print('Mesh controls:')
print('  RF                 ', MESH_RF)
print('  Patterned GND      ', MESH_PATTERNED_GND)
print('  Buried GND nominal', MESH_BURIED_GND, '(interface mesh is inherited)')
print('  SiO2               ', MESH_SIO2)
print('  Silicon            ', MESH_SILICON)
print('  Air                ', MESH_AIR)
print('Swept layers: 3 through top oxide; 4 through bottom oxide.')
print('Mesh and stationary study configured but NOT run.')


## 7. Save the unsolved setup

Run this cell after all preceding cells. Open the resulting MPH file in COMSOL, inspect the explicit `RF`, `GND_PATTERNED`, and `GND_BURIED` selections, then choose **Study 1 > Compute**. After solving, evaluate **Results > Derived Values > Zero-thickness capacitance checks**. Agreement between the terminal-charge and energy results is a necessary consistency check; repeat with mesh refinement before reporting a converged capacitance.

In [ ]:
repo_output_dir = Path.cwd() / 'Simulations' / 'trap_capacitance'
output_dir = repo_output_dir if repo_output_dir.is_dir() else Path.cwd()
setup_file = (
    output_dir / 'COMSOL_MODEL_V3_buried_GND_zero_thickness_setup.mph'
)
model.save(str(setup_file))
print('Saved configured, unsolved model:', setup_file.resolve())
print('Next: inspect selections and run Study 1 in COMSOL.')
